# Station Stacking v13 - KSEA

Experimental notebook for `KSEA`.

This version keeps the v11 remaining-warmup target and Huber/ridge stack, adds direct GFS/HRRR weather fields for rain/cloud/dewpoint/humidity, and writes artifacts to `data/calibration/station_stacking_v13`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KSEA"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v13_weather_warmup_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V13_DROPPED_FEATURE_COLUMNS,
    V13_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V13 Contract

`feature_version="v13"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V13_FEATURE_COLUMNS, sorted(V13_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
24,KSEA,gfs,1992,2021-01-01,2026-06-21
25,KSEA,hrrr,1998,2021-01-01,2026-06-21
26,KSEA,nbm,1997,2021-01-01,2026-06-21


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v13",
    target_mode="remaining_warmup",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v13",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v13/KSEA_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:2472: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2471: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2472: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,729,1.482527,2.051174
1,validation_2024_2025,lightgbm,729,1.482412,2.043328
2,validation_2024_2025,catboost,729,1.490238,2.014014
3,validation_2024_2025,provider_mean,729,2.327270,2.952714
4,validation_2024_2025,provider_median,729,2.299679,2.885702
5,validation_2024_2025,nbm_raw,729,2.344539,2.942433
6,validation_2024_2025,hrrr_raw,729,2.990762,3.865106
7,validation_2024_2025,gfs_raw,729,2.860080,3.786443
8,test_2026,xgboost,170,1.364730,1.724911
9,test_2026,lightgbm,170,1.351997,1.740859


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/station_stacking_v13",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v13/model_weights/KSEA_station_high_regressor_v13_weather_warmup_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v13/model_weights/KSEA_station_high_regressor_v13_weather_warmup_stack.json'))

## Rain-Day V11 vs V13 MAE


In [9]:
def _rain_day_flags(features: pd.DataFrame) -> pd.DataFrame:
    work = features.copy()
    work["contract_date"] = pd.to_datetime(work["contract_date"]).dt.strftime("%Y-%m-%d")
    rain_signal = pd.Series(False, index=work.index)
    candidate_cols = [
        "observed_is_raining_at_as_of",
        "v4_observed_precip_any",
        "v4_any_forecast_precip",
        "gfs_forecast_has_precip",
        "hrrr_forecast_has_precip",
        "nbm_forecast_has_precip",
    ]
    for column in candidate_cols:
        if column in work:
            rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0)
    amount_cols = [
        column
        for column in work.columns
        if column.endswith("forecast_precip_total_mm")
        or column.endswith("forecast_precip_max_1h_mm")
        or column in {"observed_precip_recent_at_as_of", "precip_amount"}
    ]
    for column in amount_cols:
        rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0.01)
    return work.loc[rain_signal, ["contract_date"]].drop_duplicates()


def _prediction_mae_by_method(predictions: pd.DataFrame, version: str, rainy_dates: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred = predictions.copy()
    pred["contract_date"] = pd.to_datetime(pred["contract_date"]).dt.strftime("%Y-%m-%d")
    pred = pred.merge(rainy_dates, on="contract_date", how="inner")
    pred = pred.dropna(subset=["actual_high_f", "predicted_high_f"])
    if pred.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred["error_f"] = pred["actual_high_f"] - pred["predicted_high_f"]
    return (
        pred.groupby("method", dropna=False)
        .agg(
            rain_day_count=("contract_date", "nunique"),
            prediction_count=("contract_date", "size"),
            mae_f=("error_f", lambda s: float(np.mean(np.abs(s)))),
            rmse_f=("error_f", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        )
        .reset_index()
        .assign(version=version)
        [["version", "method", "rain_day_count", "prediction_count", "mae_f", "rmse_f"]]
    )


v11_pred_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_year_split_test_predictions.csv"
v11_feature_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_features.csv"
v11_predictions = pd.read_csv(v11_pred_path) if v11_pred_path.exists() else pd.DataFrame()
rain_feature_frame = result.features if not result.features.empty else pd.read_csv(v11_feature_path)
rainy_dates = _rain_day_flags(rain_feature_frame)

rain_mae_comparison = pd.concat(
    [
        _prediction_mae_by_method(v11_predictions, "v11", rainy_dates),
        _prediction_mae_by_method(result.test_predictions, "v13", rainy_dates),
    ],
    ignore_index=True,
)
rain_mae_comparison = rain_mae_comparison.sort_values(["method", "version"]).reset_index(drop=True)
rain_mae_comparison


,version,method,rain_day_count,prediction_count,mae_f,rmse_f
0,v11,catboost,25,25,1.573666,1.979031
1,v13,catboost,27,27,1.533204,1.885665
2,v11,gfs_raw,25,25,2.267694,2.843427
3,v13,gfs_raw,27,27,2.372559,3.010336
4,v11,hrrr_raw,25,25,1.687063,2.131408
5,v13,hrrr_raw,27,27,1.833023,2.383216
6,v11,lightgbm,25,25,1.478400,1.826745
7,v13,lightgbm,27,27,1.576611,1.913895
8,v11,nbm_raw,25,25,1.394878,1.866420
9,v13,nbm_raw,27,27,1.390813,1.833203


In [10]:
rain_mae_wide = rain_mae_comparison.pivot_table(
    index="method",
    columns="version",
    values="mae_f",
    aggfunc="first",
)
if {"v11", "v13"}.issubset(rain_mae_wide.columns):
    rain_mae_wide["v13_minus_v11_mae_f"] = rain_mae_wide["v13"] - rain_mae_wide["v11"]
rain_mae_wide.sort_values("v13_minus_v11_mae_f" if "v13_minus_v11_mae_f" in rain_mae_wide else rain_mae_wide.columns[0])


version,v11,v13,v13_minus_v11_mae_f
method,,,
xgboost,1.394086,1.328724,-0.065362
catboost,1.573666,1.533204,-0.040462
ridge_stack,1.410938,1.390258,-0.020679
nbm_raw,1.394878,1.390813,-0.004065
lightgbm,1.478400,1.576611,0.098211
gfs_raw,2.267694,2.372559,0.104865
hrrr_raw,1.687063,1.833023,0.145960
provider_mean,NaN,1.595917,NaN
provider_median,NaN,1.650121,NaN


## V13 Feature Coverage


In [11]:
v13_feature_coverage = (
    result.features[V13_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v13_feature_coverage


,feature,coverage_pct
0,v2_morning_warmup_to_consensus_f,100.0
1,v2_humidity_warmup_interaction,100.0
2,v2_spread_per_warmup_f,100.0
3,v3_remaining_warmup_per_spread_f,100.0
4,v3_remaining_warmup_from_high_so_far_f,100.0
...,...,...
70,v13_low_cloud_cover_mean_pct,0.0
71,v13_low_visibility_flag,0.0
72,v13_low_ceiling_flag,0.0
73,v13_shortwave_mean_w_m2,0.0


In [12]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V13_FEATURE_COLUMNS)]


,feature,kind
21,observed_temp_change_last_1h_f,numeric
22,observed_temp_change_last_3h_f,numeric
23,observed_morning_warmup_rate_f_per_hour,numeric
24,observed_high_so_far_change_since_9am_f,numeric
260,v2_recent_heat_anomaly_f,numeric
...,...,...
319,climatology_high_10y_std_f,numeric
320,climatology_high_10y_count,numeric
321,provider_mean_minus_climatology_10y_f,numeric
322,observed_temp_minus_climatology_10y_f,numeric


## Dropped Feature Check


In [13]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V13_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [14]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


## Rounded Within 1F Accuracy


In [15]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,170,116,68.235294
7,oof_2026,ridge_stack,170,114,67.058824
3,oof_2026,lightgbm,170,108,63.529412
8,oof_2026,xgboost,170,107,62.941176
1,oof_2026,gfs_raw,170,80,47.058824
6,oof_2026,provider_median,170,70,41.176471
4,oof_2026,nbm_raw,170,69,40.588235
5,oof_2026,provider_mean,170,68,40.000000
2,oof_2026,hrrr_raw,170,42,24.705882
12,validation_2024_2025,lightgbm,729,460,63.100137


## Version Comparison


In [16]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
    ("v13", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v13"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,113,1.246979,1.575892,v6
1,test_2026,lightgbm,114,1.270993,1.676493,v7
2,test_2026,catboost,114,1.282843,1.677358,v7
3,test_2026,catboost,113,1.282965,1.612390,v6
4,test_2026,xgboost,136,1.292638,1.737796,v8
...,...,...,...,...,...,...
100,validation_2024_2025,hrrr_raw,589,2.854942,3.482365,v6
101,validation_2024_2025,gfs_raw,729,2.860080,3.786443,v13
102,validation_2024_2025,gfs_raw,644,2.917917,3.861741,v7
103,validation_2024_2025,hrrr_raw,729,2.990762,3.865106,v13


## 2026 OOF Weather Brackets


In [17]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,170,1.364730,1.724911,44.117647
1,lightgbm,170,1.351997,1.740859,45.294118
2,catboost,170,1.328073,1.741540,48.235294
3,ridge_stack,170,1.322149,1.697707,47.647059
4,provider_mean,170,2.078617,2.619852,27.647059
5,provider_median,170,2.127931,2.724300,25.294118
6,nbm_raw,170,2.084734,2.677866,24.117647
7,hrrr_raw,170,2.820982,3.283158,14.705882
8,gfs_raw,170,2.442151,3.549585,35.294118
